# Argo Anomaly Detection - Quick Start

This notebook demonstrates the basic usage of the Argo anomaly detection system.

In [ ]:
import sys
sys.path.append('..')

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Import system components
from data.loader import DataLoader
from features.engineer import FeatureEngineer
from features.oceanographic import OceanographicFeatures

print("Imports successful!")

## 1. Load Configuration

In [ ]:
# Load config
with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

print("Configuration loaded")
print(f"Data path: {config['paths']['data']}")

## 2. Load Sample Data

In [ ]:
# Initialize data loader
loader = DataLoader(config)

# Load one week of data
profiles = loader.load_profiles_csv(
    start_date='2019-01-01',
    end_date='2019-01-07'
)

print(f"Loaded {len(profiles)} profiles")
profiles.head()

In [ ]:
# Load measurements
measurements = loader.load_measurements_csv(
    profile_ids=profiles['global_profile_id'].unique()
)

print(f"Loaded {len(measurements)} measurements")
measurements.head()

## 3. Extract Features from a Single Profile

In [ ]:
# Get first profile
profile_id = measurements['global_profile_id'].iloc[0]
profile_data = measurements[
    measurements['global_profile_id'] == profile_id
].sort_values('pressure')

profile_meta = profiles[profiles['global_profile_id'] == profile_id].iloc[0]

print(f"Profile: {profile_id}")
print(f"Location: ({profile_meta['latitude']:.2f}, {profile_meta['longitude']:.2f})")
print(f"Date: {profile_meta['date']}")
print(f"Measurements: {len(profile_data)}")

In [ ]:
# Extract oceanographic features
temp = profile_data['temperature'].values
sal = profile_data['salinity'].values
pres = profile_data['pressure'].values
lat = profile_meta['latitude']
lon = profile_meta['longitude']

# Compute features
density = OceanographicFeatures.compute_density(temp, sal, pres, lat, lon)
mld = OceanographicFeatures.compute_mld(temp, sal, pres, lat, lon)
therm_depth = OceanographicFeatures.compute_thermocline_depth(temp, pres)

print(f"\nOceanographic Features:")
print(f"  Mixed Layer Depth: {mld:.1f} dbar")
print(f"  Thermocline Depth: {therm_depth:.1f} dbar")
print(f"  Surface Density: {density[0]:.2f} kg/m³")
print(f"  Max Depth: {pres.max():.1f} dbar")

## 4. Visualize Profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# Temperature
axes[0].plot(temp, pres, 'b-', linewidth=2)
axes[0].axhline(mld, color='r', linestyle='--', label=f'MLD: {mld:.1f}m')
axes[0].axhline(therm_depth, color='g', linestyle='--', label=f'Thermocline: {therm_depth:.1f}m')
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Pressure (dbar)')
axes[0].set_title('Temperature Profile')
axes[0].invert_yaxis()
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Salinity
axes[1].plot(sal, pres, 'g-', linewidth=2)
axes[1].set_xlabel('Salinity (PSU)')
axes[1].set_ylabel('Pressure (dbar)')
axes[1].set_title('Salinity Profile')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

# Density
axes[2].plot(density, pres, 'r-', linewidth=2)
axes[2].set_xlabel('Density (kg/m³)')
axes[2].set_ylabel('Pressure (dbar)')
axes[2].set_title('Density Profile')
axes[2].invert_yaxis()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Extract Features for Multiple Profiles

In [ ]:
# Initialize feature engineer
engineer = FeatureEngineer(config)

# Process first 100 profiles
sample_profiles = profiles.head(100)
features_df = engineer.process_profiles_batch(
    sample_profiles,
    measurements
)

print(f"Extracted features for {len(features_df)} profiles")
print(f"Number of features: {len(engineer.feature_names)}")
features_df.head()

In [ ]:
# Feature statistics
feature_stats = engineer.compute_feature_importance_stats(features_df)
print("\nTop 10 features by variance:")
feature_stats.head(10)

## 6. Next Steps

Now that you have:
1. ✅ Loaded data
2. ✅ Extracted features

You can proceed to:
- Run clustering to identify regions
- Train regional models
- Detect anomalies

See the other notebooks for these steps!